# Building a FEATHER+ Accelerator with Allo

**Hands-on Tutorial (20 min)**

In this tutorial you will:

1. Learn Allo's **dataflow streaming** primitives (`Stream`, `put`, `get`)
2. Build **spatially-parallel PE arrays** with `meta_for` and `get_pid`
3. Run a complete **FEATHER+ systolic array** simulation
4. Synthesize to **Vitis HLS** and analyze cycle-level performance

**Prerequisites:** Basic Python, familiarity with matrix multiplication. Allo is pre-installed.

In [ ]:
# ---- Run this cell first ----
import numpy as np
import allo
from allo.ir.types import int32, Stream
import allo.dataflow as df

from _support import (
    load_tutorial_trace, print_trace_summary,
    run_feather_simulation, run_feather_csynth,
    Partition,
)

print("Setup complete.")

---
## Part 1: Allo Dataflow Fundamentals

Allo's dataflow model connects **kernels** via **streams** (hardware FIFOs).
Each kernel runs as an independent hardware process.

| Primitive | What it does |
|---|---|
| `Stream[type, depth]` | Declare a FIFO of given element type and depth |
| `stream.put(value)` | Write a value (blocks if FIFO full) |
| `stream.get()` | Read a value (blocks if FIFO empty) |
| `@df.kernel(mapping=[N])` | Create N parallel hardware copies of a kernel |
| `df.get_pid()` | Get the instance ID (0..N-1) of the current copy |

```
  DRAM                                            DRAM
   |         s_a         +---------+   s_c         |
   +---> [ loader ] ---->| compute |----> [ store ] +
             |    s_b    |  (MAC)  |
             +---------->|         |
                         +---------+
```

### Exercise 1: Stream-Based Dot Product

Build a dataflow kernel that computes the **dot product** of two 4-element vectors.

- `loader` reads A and B from memory, streams values to `compute`
- `compute` reads from streams, multiplies, accumulates — the **MAC unit**
- `store` writes the result to memory

**Your task:** Fill in the `compute` kernel body (replace `pass`).

In [ ]:
K = 4

@df.region()
def dot_product(A: int32[K], B: int32[K], C: int32[1]):
    # Declare streams (FIFOs) between kernels
    s_a: Stream[int32, K]    # A values: loader -> compute
    s_b: Stream[int32, K]    # B values: loader -> compute
    s_c: Stream[int32, 1]    # Result:   compute -> store

    @df.kernel(mapping=[1], args=[A, B])
    def loader(local_A: int32[K], local_B: int32[K]):
        for i in range(K):
            s_a.put(local_A[i])
            s_b.put(local_B[i])

    @df.kernel(mapping=[1])
    def compute():
        # ============================================
        #  YOUR CODE HERE: Dot product via streams
        #  1. Initialize an accumulator (int32)
        #  2. Loop K times: read from s_a and s_b,
        #     multiply, accumulate
        #  3. Send the result to s_c via .put()
        # ============================================
        pass  # <-- Replace this

    @df.kernel(mapping=[1], args=[C])
    def store(local_C: int32[1]):
        local_C[0] = s_c.get()

# Build and test
mod = df.build(dot_product, target="simulator")
A_test = np.array([1, 2, 3, 4], dtype=np.int32)
B_test = np.array([4, 3, 2, 1], dtype=np.int32)
C_test = np.zeros(1, dtype=np.int32)
mod(A_test, B_test, C_test)

print(f"A = {A_test}")
print(f"B = {B_test}")
print(f"A . B = {C_test[0]}  (expected: {np.dot(A_test, B_test)})")
assert C_test[0] == np.dot(A_test, B_test), "FAIL"
print("PASS")

<details>
<summary><b>Click to reveal answer</b></summary>

Replace `pass` in the `compute` kernel with:

```python
        accum: int32 = 0
        for i in range(K):
            a: int32 = s_a.get()
            b: int32 = s_b.get()
            accum += a * b
        s_c.put(accum)
```

**Note:** The `loader` and `store` kernels use `args=[A, B]` / `args=[C]` to declare
which DRAM arrays they access. In Allo dataflow, every kernel that reads/writes DRAM
must list those arrays in `args=` and declare them as typed local parameters.

</details>

---
## Part 2: Spatial Parallelism

Real accelerators use **many PEs in parallel**. Allo makes this easy:

| Primitive | Effect |
|---|---|
| `@df.kernel(mapping=[N])` | N **independent hardware copies** of the kernel |
| `df.get_pid()` | Returns each copy's unique ID (0 to N-1) |
| `allo.meta_for(N) as i` | Unrolls a loop into N **parallel** hardware operations |
| `Stream[type, depth][N]` | Array of N streams — one per PE |

```
                +------+   s_c[0]
  s_a[0] ------>| PE 0 |---------->
  s_b[0] ------>|      |
                +------+
                +------+   s_c[1]
  s_a[1] ------>| PE 1 |---------->
  s_b[1] ------>|      |
                +------+
                +------+   s_c[2]
  s_a[2] ------>| PE 2 |---------->
  s_b[2] ------>|      |
                +------+
                +------+   s_c[3]
  s_a[3] ------>| PE 3 |---------->
  s_b[3] ------>|      |
                +------+
```

### Exercise 2: Parallel PE Array

Scale up to **4 parallel PEs**, each computing one element of a vector-matrix product:

$$C[4] = A[4] \times B[4 \times 4]$$

Each PE computes: `C[col] = sum_k A[k] * B[k, col]`

**Your task:** Fill in the `compute` kernel (replace `pass`).

Hints:
- `pid = df.get_pid()` tells you which PE you are (0-3)
- Read from `s_a[pid]` and `s_b[pid]`
- Output to `s_c[pid]`

In [ ]:
K, N = 4, 4

@df.region()
def parallel_mac(A: int32[K], B: int32[K, N], C: int32[N]):
    s_a: Stream[int32, K][N]     # A broadcast to each PE
    s_b: Stream[int32, K][N]     # B column per PE
    s_c: Stream[int32, 1][N]     # Output per PE

    @df.kernel(mapping=[1], args=[A, B])
    def loader(local_A: int32[K], local_B: int32[K, N]):
        for k in range(K):
            with allo.meta_for(N) as col:  # 4 parallel writes
                s_a[col].put(local_A[k])         # Same A value to all PEs
                s_b[col].put(local_B[k, col])    # Different B column per PE

    @df.kernel(mapping=[N])        # <-- 4 parallel PEs!
    def compute():
        pid = df.get_pid()
        # ============================================
        #  YOUR CODE HERE: MAC unit (5 lines)
        #  Same as Exercise 1, but use
        #    s_a[pid].get() and s_b[pid].get()
        #  Output to s_c[pid].put(...)
        # ============================================
        pass  # <-- Replace this

    @df.kernel(mapping=[1], args=[C])
    def store(local_C: int32[N]):
        with allo.meta_for(N) as col:
            local_C[col] = s_c[col].get()

# Build and test
mod = df.build(parallel_mac, target="simulator")
A_test = np.array([1, 2, 3, 4], dtype=np.int32)
B_test = np.eye(4, dtype=np.int32) * 3
C_test = np.zeros(4, dtype=np.int32)
mod(A_test, B_test, C_test)

expected = A_test @ B_test
print(f"A     = {A_test}")
print(f"B     = diagonal({B_test[0,0]})")
print(f"A x B = {C_test}  (expected: {expected})")
assert np.array_equal(C_test, expected), "FAIL"
print("PASS")

<details>
<summary><b>Click to reveal answer</b></summary>

Replace `pass` in the `compute` kernel with:

```python
        accum: int32 = 0
        for k in range(K):
            a: int32 = s_a[pid].get()
            b: int32 = s_b[pid].get()
            accum += a * b
        s_c[pid].put(accum)
```

</details>

---
## Part 3: FEATHER+ Full Accelerator

Now let's see these concepts at work in a **real accelerator**.

FEATHER+ uses a **4x4 PE array** (16 PEs) organized into 7 pipelined dataflow kernels:

```
  DRAM                                                 DRAM
   |                                                    ^
   v                                                    |
 +----------+  col_a_in   +---------------------+   +---------------+
 | a_loader |------------>|                     |   |               |
 +----------+             |   pe_array [5 x 4]  |   |               |
                          |                     |   |               |
 +----------+  col_w_in   |  +--+--+--+--+     |   |               |
 | w_loader |------+      |  |PE|PE|PE|PE| row0 |   |               |
 +----------+      |      |  +--+--+--+--+     |   |               |
              +-----v---+ |  |PE|PE|PE|PE| row1 |   | output_accum  |
              |w_broad- | |  +--+--+--+--+     |   |               |
              |cast [4] |--->|PE|PE|PE|PE| row2 |   |               |
              +---------+ |  +--+--+--+--+     |   |               |
                          |  |PE|PE|PE|PE| row3 |   |               |
                          |  +--+--+--+--+     |   |               |
                          |  |G |G |G |G | gath.|   |               |
                          |  +--+--+--+--+     |   |               |
                          +--------+-----------+   |               |
                                   |               |               |
                          +--------v-----------+   |               |
   +---------+  inst      |  BIRRD [3 x 2]    |   |               |
   | inst_rw |----------->| (butterfly reduce) |-->|               |
   +---------+            +--------------------+   +-------+-------+
                                                           |
                                                           v
                                                        C [M, N]
```

**Key patterns you already learned:**
- Each PE uses `stream.get()` / `stream.put()` (Exercise 1)
- The PE array uses `mapping=[5, 4]` for 20 parallel instances (Exercise 2)
- `meta_if` / `meta_else` specializes behavior per PE row

### FEATHER+ PE Array — Key Code

Here is the PE array kernel, using the same patterns from the exercises:

```python
@df.kernel(mapping=[AH + 1, AW])     # 5x4 = 20 instances
def pe_array():
    ni, nj = df.get_pid()             # (row, col) of this PE

    with allo.meta_if(ni == AH):      # Row 4: Gather unit
        # Collect results from compute PEs, send to BIRRD
        ...

    with allo.meta_else():            # Rows 0-3: Compute PEs
        for _op in range(total_ops):
            tile_accum: int32 = 0
            for nk in range(AH):
                # Column-streaming: row 0 reads from DRAM loader,
                # rows 1+ read from the PE above
                a_val: int32 = 0
                with allo.meta_if(ni == 0):
                    a_val = col_a_in[nj].get()           # From a_loader
                with allo.meta_else():
                    a_val = pe_a_down[ni - 1, nj].get()  # From PE above

                w_val: int32 = pe_w_in[ni, nj].get()     # From w_broadcast

                # Forward A to next row (column-streaming)
                with allo.meta_if(ni < AH - 1):
                    pe_a_down[ni, nj].put(a_val)

                tile_accum += a_val * w_val  # <-- MAC!

            pe_out[ni, nj].put(tile_accum)
```

Notice how `meta_if(ni == 0)` generates **different hardware** for row 0 vs other rows,
while `tile_accum += a_val * w_val` is the same MAC you wrote in Exercise 1.

### Run FEATHER+ Simulation

Let's run the full 4x4 FEATHER+ on a real GEMM workload from the paper:

**Figure 7:** `C[16, 8] = A[16, 12] x B[12, 8]`

In [ ]:
# Load the workload trace
trace = load_tutorial_trace()
print_trace_summary(trace)

In [ ]:
# Build and run the full FEATHER+ simulation (~10s)
C_result, passed = run_feather_simulation(trace)

---
## Part 4: HLS Synthesis

To turn our Allo design into **real hardware**, we synthesize with Vitis HLS.

The key optimization is **scheduling** — telling HLS how to pipeline loops
and partition arrays for parallel access:

| Directive | Effect |
|---|---|
| `s.pipeline("kernel:loop")` | Pipeline a loop for maximum throughput |
| `s.partition("kernel:array", dim=N, ...)` | Split an array into parallel banks |
| `Partition.Complete` | Every element gets its own port (registers) |
| `Partition.Cyclic` | Round-robin distribution across banks |

### Exercise 3: Write the HLS Schedule

Fill in the scheduling directives to optimize the FEATHER+ kernel.

**What to optimize:**
1. **Pipeline** the `w_loader`'s tile loop so tiles overlap
2. **Partition** the output C along columns for parallel writes
3. **Partition** input A along rows for AW parallel reads per cycle
4. **Partition** weight B completely (small enough for registers)
5. **Partition** the accumulator buffer for parallel access

In [ ]:
def my_schedule(s, K, N, AH, AW):
    """HLS scheduling directives for FEATHER+."""
    # ============================================
    #  YOUR CODE HERE
    # ============================================

    # 1. Pipeline the w_loader tile loop
    #    s.pipeline("w_loader_0:tile")

    # 2. Partition C along dim=2 for parallel output writes
    #    s.partition("full_matrix_top:C", dim=2, factor=AH)

    # 3. Partition A along dim=1 for AW parallel row reads
    #    s.partition("a_loader_0:local_A", dim=1, factor=AW,
    #                partition_type=Partition.Cyclic)

    # 4. Partition B completely (both dims) for register storage
    #    s.partition("w_loader_0:local_B", dim=1,
    #                partition_type=Partition.Complete)
    #    s.partition("w_loader_0:local_B", dim=2,
    #                partition_type=Partition.Complete)

    # 5. Partition accumulator buffer
    #    s.partition("output_accum_0:local_acc", dim=1,
    #                partition_type=Partition.Complete)
    #    s.partition("output_accum_0:local_acc", dim=2,
    #                partition_type=Partition.Complete)

    pass  # <-- Remove this after uncommenting the directives above

<details>
<summary><b>Click to reveal answer</b></summary>

Uncomment all the directives and remove `pass`:

```python
def my_schedule(s, K, N, AH, AW):
    s.pipeline("w_loader_0:tile")
    s.partition("full_matrix_top:C", dim=2, factor=AH)
    s.partition("a_loader_0:local_A", dim=1, factor=AW,
                partition_type=Partition.Cyclic)
    s.partition("w_loader_0:local_B", dim=1,
                partition_type=Partition.Complete)
    s.partition("w_loader_0:local_B", dim=2,
                partition_type=Partition.Complete)
    s.partition("output_accum_0:local_acc", dim=1,
                partition_type=Partition.Complete)
    s.partition("output_accum_0:local_acc", dim=2,
                partition_type=Partition.Complete)
```

</details>

In [ ]:
# Run HLS synthesis (requires Vitis HLS, ~2 minutes)
run_feather_csynth(trace, my_schedule)

---
## Summary

In this tutorial you built a FEATHER+ accelerator using Allo:

| Concept | Allo Primitive | Where |
|---|---|---|
| Dataflow streaming | `Stream`, `.put()`, `.get()` | Exercise 1 |
| Spatial parallelism | `mapping=[N]`, `meta_for(N)` | Exercise 2 |
| Compile-time specialization | `meta_if` / `meta_else` | PE array walkthrough |
| HLS scheduling | `pipeline`, `partition` | Exercise 3 |

The full 4x4 FEATHER+ achieves **~600 cycles** for a 16x12x8 GEMM.
The same architecture scales to **16x16 arrays** for production workloads.

**Next steps:**
- Try different `Partition` strategies and see how they affect cycle count
- Look at `feather_minisa.py` for the full kernel source code
- Run RTL co-simulation with `--hls cosim` for cycle-accurate verification